In [ ]:
import gymnasium as gym
import numpy as np
import random
import time


env = gym.make("CliffWalking-v1", is_slippery=False, render_mode="human")

n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

alpha = 0.8
gamma = 0.95
epsilon = 1.0
epsilon_decay = 0.999  
epsilon_min = 0.01

episodes = 3000
max_steps = 100


train_env = gym.make("CliffWalking-v1", is_slippery=False)

print("Starting Training...")

total_successes = 0

for episode in range(episodes):

    state, _ = train_env.reset()

    for step in range(max_steps):

        # Epsilon Greedy
        if random.uniform(0,1) < epsilon:
            action = train_env.action_space.sample()
        else:
            action = np.argmax(Q[state])

        next_state, reward, terminated, truncated, _ = train_env.step(action)

        # Q Update
        Q[state, action] = Q[state, action] + alpha * (
            reward + gamma * np.max(Q[next_state]) - Q[state, action]
        )

        state = next_state

        if terminated or truncated:
            if reward == 1.0:  # Reached the goal
                total_successes += 1
            break

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    if (episode + 1) % 500 == 0:
        print(f"Episode {episode + 1}/{episodes} completed. Epsilon: {epsilon:.4f}, Successes so far: {total_successes}")
        print("Current Q-Table:")
        print(Q)

train_env.close()

print(f"\nTraining Completed! Total Successes: {total_successes}/{episodes}")
print("Final Q-Table:")
print(Q)

Starting Training...
Episode 500/3000 completed. Epsilon: 0.6064, Successes so far: 0
Current Q-Table:
[[ -10.7341754   -10.24650042  -10.24650042  -10.7341754 ]
 [ -10.24650042   -9.73315833   -9.73315833  -10.7341754 ]
 [  -9.73315833   -9.19279825   -9.19279825  -10.24650042]
 [  -9.19279825   -8.62399815   -8.62399815   -9.73315833]
 [  -8.62399815   -8.02526122   -8.02526122   -9.19279825]
 [  -8.02526122   -7.39501181   -7.39501181   -8.62399815]
 [  -7.39501181   -6.73159137   -6.73159137   -8.02526122]
 [  -6.73159137   -6.03325408   -6.03325408   -7.39501181]
 [  -6.03325408   -5.29816219   -5.29816219   -6.73159137]
 [  -5.29816219   -4.52438125   -4.52438125   -6.03325408]
 [  -4.52438125   -3.709875     -3.709875     -5.29816219]
 [  -3.709875     -3.709875     -2.8525       -4.52438125]
 [ -10.7341754    -9.73315833   -9.73315833  -10.24650042]
 [ -10.24650042   -9.19279825   -9.19279825  -10.24650042]
 [  -9.73315833   -8.62399815   -8.62399815   -9.73315833]
 [  -9.19279

In [10]:
print("\nStarting Visualization...")

state, _ = env.reset()

done = False
while not done:
    action = np.argmax(Q[state])
    state, reward, terminated, truncated, _ = env.step(action)
    time.sleep(0.5)  # Slow down so you can see movement
    done = terminated or truncated

env.close()


Starting Visualization...


In [3]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. ENVIRONMENT SETUP
# ============================================================
class GridWorld2D(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, grid_size=5, render_mode=None):
        super().__init__()
        self.grid_size = grid_size
        self.render_mode = render_mode
        self.action_space = spaces.Discrete(4) # 0=Up, 1=Down, 2=Left, 3=Right
        self.observation_space = spaces.Box(low=0, high=grid_size-1, shape=(2,), dtype=np.int32)
        self.goal_pos = np.array([grid_size-1, grid_size-1])
        
        # Define Holes and Obstacles
        self.holes = {(1, 3), (3, 1)}
        self.obstacles = {(1, 1), (2, 2)}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.agent_pos = np.array([0, 0], dtype=np.int32)
        return self.agent_pos, {}

    def step(self, action):
        x, y = self.agent_pos
        nx, ny = x, y

        # Action logic (0=Up, 1=Down, 2=Left, 3=Right)
        if action == 0: nx -= 1
        elif action == 1: nx += 1
        elif action == 2: ny -= 1
        elif action == 3: ny += 1

        nx = np.clip(nx, 0, self.grid_size - 1)
        ny = np.clip(ny, 0, self.grid_size - 1)

        if (nx, ny) in self.obstacles:
            nx, ny = x, y

        self.agent_pos = np.array([nx, ny])
        terminated = False
        
        if (nx, ny) in self.holes:
            reward = -10
            terminated = True
        elif np.array_equal(self.agent_pos, self.goal_pos):
            reward = 10
            terminated = True
        else:
            reward = -1
            
        return self.agent_pos, reward, terminated, False, {}

# ============================================================
# 2. TRAINING (Q-LEARNING)
# ============================================================
def run_training():
    env = GridWorld2D(grid_size=5)
    grid_size = 5
    n_states = grid_size * grid_size
    n_actions = 4
    Q = np.zeros((n_states, n_actions))
    
    # Hyperparameters
    episodes = 2000  # Increased slightly to ensure convergence
    alpha = 0.1
    gamma = 0.99
    epsilon = 1.0
    epsilon_decay = 0.995
    epsilon_min = 0.05

    for _ in range(episodes):
        state, _ = env.reset()
        state_idx = state[0] * grid_size + state[1]
        done = False

        while not done:
            if np.random.rand() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(Q[state_idx])

            next_state, reward, terminated, _, _ = env.step(action)
            next_idx = next_state[0] * grid_size + next_state[1]
            
            # Q-Update
            best_next = np.max(Q[next_idx])
            Q[state_idx, action] += alpha * (reward + gamma * best_next - Q[state_idx, action])
            
            state_idx = next_idx
            done = terminated

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    return Q

# ============================================================
# 3. VISUALIZATION
# ============================================================
def print_simple_visuals(env, Q):
    grid_size = env.grid_size
    
    # Symbols
    arrows = ["↑", "↓", "←", "→"]
    
    print("\n" + "="*30)
    print("1. BEST PATH (POLICY)")
    print("="*30)

    for r in range(grid_size):
        row_string = ""
        for c in range(grid_size):
            state_idx = r * grid_size + c
            
            # Check for special map items
            if (r, c) == (0, 0):
                char = "S" # Start
            elif (r, c) == (grid_size - 1, grid_size - 1):
                char = "G" # Goal
            elif (r, c) in env.obstacles:
                char = "X" # Wall
            elif (r, c) in env.holes:
                char = "H" # Hole
            else:
                # Decide best action based on Q-table
                best_action = np.argmax(Q[state_idx])
                char = arrows[best_action]
            
            row_string += f" {char} "
        print(row_string)

    print("\n" + "="*30)
    print("2. VALUE MAP (HEATMAP)")
    print("="*30)

    for r in range(grid_size):
        row_string = ""
        for c in range(grid_size):
            state_idx = r * grid_size + c
            
            # Get the max value for this state (how good is it to be here?)
            max_val = np.max(Q[state_idx])
            
            # Formatting: Round to integer for clean reading
            if (r, c) in env.obstacles:
                row_string += "  X "
            elif (r, c) in env.holes:
                row_string += "  H "
            else:
                row_string += f"{max_val:3.0f} " # Formats as width 3 integer
        print(row_string)
        
if __name__ == "__main__":
    # 1. Create the environment instance here so it's globally accessible
    env = GridWorld2D(grid_size=5)
    
    # 2. Run the training
    Q_table = run_training()
    
    # 3. Pass the environment and the Q_table to the visuals
    print_simple_visuals(env, Q_table)



1. BEST PATH (POLICY)
 S  →  →  →  ↓ 
 ↑  X  ↑  H  ↓ 
 ↓  ↑  X  →  ↓ 
 ↓  H  →  →  ↓ 
 →  →  →  →  G 

2. VALUE MAP (HEATMAP)
  3   4   5   6   7 
  1   X   3   H   8 
 -3  -3   X   8   9 
 -1   H   3   9  10 
  1   4   7   9   0 
